# PS S6E6 - 007 Bias search on 002 LightGBM raw baseline, blend-ready

Experiment: `exp_20260603_007_lgb002_bias_search`

Purpose:
- Load 002 LightGBM strict raw OOF/test probabilities.
- Search class-wise bias on OOF to maximize `balanced_accuracy_score`.
- Apply the best bias to both OOF and test log-probabilities.
- Save adjusted OOF/test probabilities as `.npy` so this experiment can be added to later blend notebooks.

Important:
- No retraining.
- No original dataset.
- No feature engineering.
- GALAXY bias is fixed to 0.
- QSO / STAR biases are searched.
- The original 002 OOF/pred probabilities are not overwritten.

In [1]:
# ============================================================
# 0. Imports / Config
# ============================================================

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260603_007_lgb002_bias_search"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")

WORKDIR = Path("/kaggle/working")
OUTDIR = WORKDIR / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

REF_002_EXP_ID = "exp_20260601_002_lgb_strict_raw_baseline"
REF_002_CV = 0.9569063401002502
REF_002_PUBLIC_LB = 0.95800

# Default paths when running in the same Kaggle working session.
# If previous notebook outputs were added as Kaggle input datasets,
# resolve_path() will search by filename under /kaggle/input.
OOF_002_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_lgb_strict_raw_proba.npy"
PRED_002_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/pred_lgb_strict_raw_proba.npy"

# Blend-ready adjusted probability filenames.
# Rule: include full exp_* id in oof_*.npy and pred_*.npy.
OOF_BIASED_NPY = OUTDIR / "oof_exp_20260603_007_lgb002_bias_search_proba.npy"
PRED_BIASED_NPY = OUTDIR / "pred_exp_20260603_007_lgb002_bias_search_proba.npy"

print("OUTDIR:", OUTDIR)
print("OOF_BIASED_NPY:", OOF_BIASED_NPY)
print("PRED_BIASED_NPY:", PRED_BIASED_NPY)

OUTDIR: /kaggle/working/exp_20260603_007_lgb002_bias_search
OOF_BIASED_NPY: /kaggle/working/exp_20260603_007_lgb002_bias_search/oof_exp_20260603_007_lgb002_bias_search_proba.npy
PRED_BIASED_NPY: /kaggle/working/exp_20260603_007_lgb002_bias_search/pred_exp_20260603_007_lgb002_bias_search_proba.npy


In [2]:
# ============================================================
# 1. Utilities
# ============================================================

def save_json(obj, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


def resolve_path(path_like: str) -> Path:
    """Resolve OOF/pred file path.

    Search order:
    1. exact path
    2. /kaggle/working/**/filename
    3. /kaggle/input/**/filename

    This supports both:
    - same-session notebook chaining
    - adding previous outputs as Kaggle datasets
    """
    p = Path(path_like)
    if p.exists():
        return p

    fname = p.name
    matches = []
    for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
        if root.exists():
            matches.extend(root.rglob(fname))

    matches = sorted(set(matches))
    if len(matches) == 1:
        print(f"[resolve_path] {path_like} -> {matches[0]}")
        return matches[0]
    if len(matches) > 1:
        print(f"[resolve_path] Multiple candidates for {fname}:")
        for m in matches[:20]:
            print(" -", m)
        print("[resolve_path] Using first candidate:", matches[0])
        return matches[0]

    raise FileNotFoundError(
        f"Could not resolve path: {path_like}\n"
        f"Expected filename: {fname}\n"
        "Add 002 outputs as Kaggle input dataset or edit OOF_002_PATH/PRED_002_PATH."
    )


def proba_to_logp(proba: np.ndarray, eps: float = 1e-15) -> np.ndarray:
    return np.log(np.clip(proba, eps, 1.0))


def softmax_np(x: np.ndarray, axis: int = 1) -> np.ndarray:
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)


def apply_bias_to_proba(proba: np.ndarray, bias: np.ndarray) -> np.ndarray:
    """Apply additive class bias in log-probability space and return normalized probabilities."""
    logp = proba_to_logp(proba)
    return softmax_np(logp + bias.reshape(1, -1), axis=1)


def predict_with_bias(logp: np.ndarray, bias: np.ndarray) -> np.ndarray:
    return np.argmax(logp + bias.reshape(1, -1), axis=1)


def class_recalls(y_true: np.ndarray, y_pred: np.ndarray, class_names: list[str]) -> dict:
    out = {}
    for i, cls in enumerate(class_names):
        m = y_true == i
        out[f"recall_{cls}"] = float((y_pred[m] == i).mean()) if m.any() else float("nan")
    return out


def eval_pred(y_true: np.ndarray, y_pred: np.ndarray, class_names: list[str]) -> dict:
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred))}
    res.update(class_recalls(y_true, y_pred, class_names))
    return res


def validate_proba(name: str, arr: np.ndarray, n_rows: int, n_classes: int) -> None:
    assert arr.shape == (n_rows, n_classes), (name, arr.shape, n_rows, n_classes)
    assert np.isfinite(arr).all(), name
    row_sum = arr.sum(axis=1)
    assert np.allclose(row_sum, 1.0, atol=1e-5), (name, row_sum.min(), row_sum.max())
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())

In [3]:
# ============================================================
# 2. Load train/test/sample and 002 predictions
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)

assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof_path = resolve_path(OOF_002_PATH)
pred_path = resolve_path(PRED_002_PATH)

oof_proba_raw = np.load(oof_path).astype(np.float32)
test_proba_raw = np.load(pred_path).astype(np.float32)

validate_proba("oof_proba_raw", oof_proba_raw, len(train), n_classes)
validate_proba("test_proba_raw", test_proba_raw, len(test), n_classes)

oof_logp = proba_to_logp(oof_proba_raw)
test_logp = proba_to_logp(test_proba_raw)

print("train:", train.shape)
print("test:", test.shape)
print("class_names:", class_names)
print("oof_path:", oof_path)
print("pred_path:", pred_path)
print("oof_proba_raw:", oof_proba_raw.shape)
print("test_proba_raw:", test_proba_raw.shape)

train: (577347, 12)
test: (247435, 11)
class_names: ['GALAXY', 'QSO', 'STAR']
oof_path: /kaggle/input/datasets/mizushimatoshihiko/npy-files/oof_lgb_strict_raw_proba.npy
pred_path: /kaggle/input/datasets/mizushimatoshihiko/npy-files/pred_lgb_strict_raw_proba.npy
oof_proba_raw: (577347, 3)
test_proba_raw: (247435, 3)


In [4]:
# ============================================================
# 3. Raw 002 confirmation
# ============================================================

raw_bias = np.zeros(n_classes, dtype=np.float64)
raw_pred = predict_with_bias(oof_logp, raw_bias)
raw_eval = eval_pred(y, raw_pred, class_names)

print("Raw 002:")
print(json.dumps(raw_eval, indent=2, ensure_ascii=False))
print("delta vs REF_002_CV:", raw_eval["balanced_accuracy"] - REF_002_CV)
print(confusion_matrix(y, raw_pred))
print(classification_report(y, raw_pred, target_names=class_names))

Raw 002:
{
  "balanced_accuracy": 0.9569063401002502,
  "recall_GALAXY": 0.978430645332203,
  "recall_QSO": 0.9650000426828748,
  "recall_STAR": 0.9272883322856729
}
delta vs REF_002_CV: 0.0
[[369338   3346   4796]
 [  3103 113043    997]
 [  5506    509  76709]]
              precision    recall  f1-score   support

      GALAXY       0.98      0.98      0.98    377480
         QSO       0.97      0.97      0.97    117143
        STAR       0.93      0.93      0.93     82724

    accuracy                           0.97    577347
   macro avg       0.96      0.96      0.96    577347
weighted avg       0.97      0.97      0.97    577347



In [5]:
# ============================================================
# 4. Bias search
# ============================================================

def grid_search_bias_2d(logp, y_true, qso_values, star_values, stage_name):
    rows = []
    best_score = -1.0
    best_bias = None

    for bq in qso_values:
        for bs in star_values:
            bias = np.array([0.0, float(bq), float(bs)], dtype=np.float64)
            pred = predict_with_bias(logp, bias)
            score = float(balanced_accuracy_score(y_true, pred))

            rows.append({
                "stage": stage_name,
                "bias_GALAXY": 0.0,
                "bias_QSO": float(bq),
                "bias_STAR": float(bs),
                "balanced_accuracy": score,
            })

            if score > best_score:
                best_score = score
                best_bias = bias.copy()

        print(f"{stage_name}: bq={bq:.6f}, best={best_score:.9f}, bias={best_bias}")

    return {"score": best_score, "bias": best_bias}, pd.DataFrame(rows)


# Stage 1 coarse
coarse_qso = np.round(np.arange(-0.120, 0.120 + 1e-12, 0.010), 6)
coarse_star = np.round(np.arange(-0.120, 0.120 + 1e-12, 0.010), 6)
best_coarse, coarse_df = grid_search_bias_2d(oof_logp, y, coarse_qso, coarse_star, "coarse")

# Stage 2 local
cq, cs = best_coarse["bias"][1], best_coarse["bias"][2]
local_qso = np.round(np.arange(cq - 0.020, cq + 0.020 + 1e-12, 0.002), 6)
local_star = np.round(np.arange(cs - 0.020, cs + 0.020 + 1e-12, 0.002), 6)
best_local, local_df = grid_search_bias_2d(oof_logp, y, local_qso, local_star, "local")

# Stage 3 ultra local
uq, us = best_local["bias"][1], best_local["bias"][2]
ultra_qso = np.round(np.arange(uq - 0.004, uq + 0.004 + 1e-12, 0.0005), 6)
ultra_star = np.round(np.arange(us - 0.004, us + 0.004 + 1e-12, 0.0005), 6)
best_ultra, ultra_df = grid_search_bias_2d(oof_logp, y, ultra_qso, ultra_star, "ultra_local")

search_df = pd.concat([coarse_df, local_df, ultra_df], ignore_index=True)
search_df = search_df.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)

best_bias = best_ultra["bias"]
best_score = best_ultra["score"]

display(search_df.head(20))
print("raw_score:", raw_eval["balanced_accuracy"])
print("best_score:", best_score)
print("delta_vs_raw:", best_score - raw_eval["balanced_accuracy"])
print("best_bias:", best_bias)

coarse: bq=-0.120000, best=0.957698517, bias=[ 0.   -0.12  0.12]
coarse: bq=-0.110000, best=0.957733135, bias=[ 0.   -0.11  0.12]
coarse: bq=-0.100000, best=0.957748698, bias=[ 0.   -0.1   0.12]
coarse: bq=-0.090000, best=0.957793632, bias=[ 0.   -0.09  0.12]
coarse: bq=-0.080000, best=0.957833070, bias=[ 0.   -0.08  0.12]
coarse: bq=-0.070000, best=0.957860232, bias=[ 0.   -0.07  0.12]
coarse: bq=-0.060000, best=0.957899069, bias=[ 0.   -0.06  0.12]
coarse: bq=-0.050000, best=0.957937324, bias=[ 0.   -0.05  0.12]
coarse: bq=-0.040000, best=0.957956033, bias=[ 0.   -0.04  0.12]
coarse: bq=-0.030000, best=0.958000280, bias=[ 0.   -0.03  0.12]
coarse: bq=-0.020000, best=0.958027742, bias=[ 0.   -0.02  0.12]
coarse: bq=-0.010000, best=0.958036915, bias=[ 0.   -0.01  0.12]
coarse: bq=-0.000000, best=0.958075948, bias=[ 0.   -0.    0.12]
coarse: bq=0.010000, best=0.958112241, bias=[0.   0.01 0.12]
coarse: bq=0.020000, best=0.958127908, bias=[0.   0.02 0.12]
coarse: bq=0.030000, best=0.95813

,stage,bias_GALAXY,bias_QSO,bias_STAR,balanced_accuracy
0,ultra_local,0.0,0.1440,0.1435,0.958610
1,ultra_local,0.0,0.1435,0.1440,0.958609
2,ultra_local,0.0,0.1435,0.1435,0.958609
3,ultra_local,0.0,0.1405,0.1435,0.958607
4,ultra_local,0.0,0.1440,0.1440,0.958606
5,ultra_local,0.0,0.1440,0.1430,0.958606
6,ultra_local,0.0,0.1440,0.1425,0.958606
7,ultra_local,0.0,0.1400,0.1435,0.958605
8,ultra_local,0.0,0.1425,0.1435,0.958605
9,ultra_local,0.0,0.1430,0.1435,0.958605


raw_score: 0.9569063401002502
best_score: 0.9586098708578418
delta_vs_raw: 0.0017035307575915537
best_bias: [0.     0.144  0.1435]


In [6]:
# ============================================================
# 5. Build blend-ready biased OOF/test probabilities
# ============================================================

# This is the key difference from the previous 007 notebook.
# We create adjusted probability matrices, not only biased labels.
# These .npy files can be loaded by later blend notebooks.
oof_proba_biased = apply_bias_to_proba(oof_proba_raw, best_bias).astype(np.float32)
test_proba_biased = apply_bias_to_proba(test_proba_raw, best_bias).astype(np.float32)

validate_proba("oof_proba_biased", oof_proba_biased, len(train), n_classes)
validate_proba("test_proba_biased", test_proba_biased, len(test), n_classes)

biased_oof_pred = oof_proba_biased.argmax(axis=1)
biased_eval = eval_pred(y, biased_oof_pred, class_names)

print("Biased:")
print(json.dumps(biased_eval, indent=2, ensure_ascii=False))
print("delta vs raw:", biased_eval["balanced_accuracy"] - raw_eval["balanced_accuracy"])
print("delta vs REF_002_CV:", biased_eval["balanced_accuracy"] - REF_002_CV)
print(confusion_matrix(y, biased_oof_pred))
print(classification_report(y, biased_oof_pred, target_names=class_names))

compare_eval = pd.DataFrame([
    {"version": "raw_002", **raw_eval},
    {"version": "biased_007", **biased_eval},
])
display(compare_eval)

print("OOF proba raw/biased diff abs mean:", float(np.mean(np.abs(oof_proba_biased - oof_proba_raw))))
print("TEST proba raw/biased diff abs mean:", float(np.mean(np.abs(test_proba_biased - test_proba_raw))))

Biased:
{
  "balanced_accuracy": 0.9586098708578418,
  "recall_GALAXY": 0.9766928049168168,
  "recall_QSO": 0.9666988210989986,
  "recall_STAR": 0.9324379865577099
}
delta vs raw: 0.0017035307575915537
delta vs REF_002_CV: 0.0017035307575915537
[[368682   3541   5257]
 [  2898 113242   1003]
 [  5073    516  77135]]
              precision    recall  f1-score   support

      GALAXY       0.98      0.98      0.98    377480
         QSO       0.97      0.97      0.97    117143
        STAR       0.92      0.93      0.93     82724

    accuracy                           0.97    577347
   macro avg       0.96      0.96      0.96    577347
weighted avg       0.97      0.97      0.97    577347



,version,balanced_accuracy,recall_GALAXY,recall_QSO,recall_STAR
0,raw_002,0.956906,0.978431,0.965000,0.927288
1,biased_007,0.958610,0.976693,0.966699,0.932438


OOF proba raw/biased diff abs mean: 0.002016583224758506
TEST proba raw/biased diff abs mean: 0.002038917038589716


In [7]:
# ============================================================
# 6. Biased submission
# ============================================================

biased_test_pred = test_proba_biased.argmax(axis=1)
biased_test_labels = le.inverse_transform(biased_test_pred)

submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET_COL: biased_test_labels,
})

assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

print(submission[TARGET_COL].value_counts(normalize=True))
display(submission.head())

submission_path = OUTDIR / "submission_lgb002_bias_search.csv"
submission.to_csv(submission_path, index=False)
print("saved:", submission_path)

class
GALAXY    0.652988
QSO       0.202958
STAR      0.144054
Name: proportion, dtype: float64


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


saved: /kaggle/working/exp_20260603_007_lgb002_bias_search/submission_lgb002_bias_search.csv


In [8]:
# ============================================================
# 7. Save artifacts
# ============================================================

np.save(OUTDIR / "bias_vector.npy", best_bias)

# Blend-ready .npy files with exp_* in filenames.
np.save(OOF_BIASED_NPY, oof_proba_biased)
np.save(PRED_BIASED_NPY, test_proba_biased)

# Optional raw copies with explicit exp name.
# These are useful for audit, but blend should normally use the biased files above.
np.save(OUTDIR / "oof_exp_20260603_007_lgb002_bias_search_raw002_proba.npy", oof_proba_raw)
np.save(OUTDIR / "pred_exp_20260603_007_lgb002_bias_search_raw002_proba.npy", test_proba_raw)

oof_df = pd.DataFrame({
    ID_COL: train[ID_COL].values,
    "y_true": train[TARGET_COL].astype(str).values,
    "raw_pred": le.inverse_transform(raw_pred),
    "biased_pred": le.inverse_transform(biased_oof_pred),
})
for i, cls in enumerate(class_names):
    oof_df[f"raw_proba_{cls}"] = oof_proba_raw[:, i]
    oof_df[f"biased_proba_{cls}"] = oof_proba_biased[:, i]
    oof_df[f"logp_{cls}"] = oof_logp[:, i]
oof_df.to_csv(OUTDIR / "oof_lgb002_bias_search.csv", index=False)

pred_df = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    "biased_pred_label": biased_test_labels,
})
for i, cls in enumerate(class_names):
    pred_df[f"raw_proba_{cls}"] = test_proba_raw[:, i]
    pred_df[f"biased_proba_{cls}"] = test_proba_biased[:, i]
    pred_df[f"logp_{cls}"] = test_logp[:, i]
pred_df.to_csv(OUTDIR / "pred_lgb002_bias_search.csv", index=False)

search_df.to_csv(OUTDIR / "bias_search_results.csv", index=False)
compare_eval.to_csv(OUTDIR / "raw_vs_biased_oof_eval.csv", index=False)

cv_result = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "model": "LightGBM 002 postprocess",
    "status": "completed",
    "metric": "balanced_accuracy_score",
    "reference_exp_id": REF_002_EXP_ID,
    "raw_cv_score": float(raw_eval["balanced_accuracy"]),
    "biased_cv_score": float(biased_eval["balanced_accuracy"]),
    "delta_biased_minus_raw": float(biased_eval["balanced_accuracy"] - raw_eval["balanced_accuracy"]),
    "reference_002_public_lb": REF_002_PUBLIC_LB,
    "class_names": class_names,
    "label_mapping": {str(cls): int(i) for i, cls in enumerate(class_names)},
    "bias": {
        "GALAXY": float(best_bias[0]),
        "QSO": float(best_bias[1]),
        "STAR": float(best_bias[2]),
    },
    "raw_eval": raw_eval,
    "biased_eval": biased_eval,
    "use_original": False,
    "use_fe": False,
    "use_bias_search": True,
    "adjusted_probability_method": "softmax(log(raw_proba) + class_bias)",
    "submission_path": str(submission_path),
    "oof_path": str(OOF_BIASED_NPY),
    "pred_path": str(PRED_BIASED_NPY),
    "raw_002_oof_path": str(oof_path),
    "raw_002_pred_path": str(pred_path),
}

save_json(cv_result, OUTDIR / "cv_result.json")

print("Artifacts saved to:", OUTDIR)
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)

Artifacts saved to: /kaggle/working/exp_20260603_007_lgb002_bias_search
 - bias_search_results.csv
 - bias_vector.npy
 - cv_result.json
 - oof_exp_20260603_007_lgb002_bias_search_proba.npy
 - oof_exp_20260603_007_lgb002_bias_search_raw002_proba.npy
 - oof_lgb002_bias_search.csv
 - pred_exp_20260603_007_lgb002_bias_search_proba.npy
 - pred_exp_20260603_007_lgb002_bias_search_raw002_proba.npy
 - pred_lgb002_bias_search.csv
 - raw_vs_biased_oof_eval.csv
 - submission_lgb002_bias_search.csv


In [9]:
# ============================================================
# 8. Registry and memo
# ============================================================

registry_path = WORKDIR / "oof_registry.csv"

registry_row = {
    "exp_id": EXP_ID,
    "model_family": "LightGBM",
    "feature_family": "strict_raw",
    "cv_metric": "balanced_accuracy_score",
    "cv_score": float(biased_eval["balanced_accuracy"]),
    "raw_cv_score": float(raw_eval["balanced_accuracy"]),
    "cv_delta_vs_raw": float(biased_eval["balanced_accuracy"] - raw_eval["balanced_accuracy"]),
    "public_lb": np.nan,
    "use_original": False,
    "use_fe": False,
    "use_bias_search": True,
    "oof_path": str(OOF_BIASED_NPY),
    "pred_path": str(PRED_BIASED_NPY),
    "submission_path": str(submission_path),
    "role": "postprocess_bias_candidate",
    "status": "completed",
    "keep_hold_drop": "pending_public_lb",
    "notes": "Class-wise bias search on 002 LGB raw. Blend-ready adjusted OOF/pred proba saved with exp_* filenames.",
}

if registry_path.exists():
    registry = pd.read_csv(registry_path)
    registry = registry[registry["exp_id"] != EXP_ID]
    registry = pd.concat([registry, pd.DataFrame([registry_row])], ignore_index=True)
else:
    registry = pd.DataFrame([registry_row])

registry.to_csv(registry_path, index=False)
registry.to_csv(OUTDIR / "oof_registry.csv", index=False)

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "Bias search on 002 LightGBM strict raw baseline, blend-ready",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "model": "LightGBM 002 postprocess",
        "created_at": "2026-06-03",
    },
    "objective": {
        "summary": (
            "Apply class-wise bias search to 002 LightGBM strict raw OOF/test probabilities. "
            "Do not retrain. GALAXY bias is fixed to 0 and QSO/STAR biases are searched. "
            "Save adjusted OOF/test probability matrices so 007 can be added to blend notebooks."
        ),
        "success_criteria": [
            "load 002 OOF and test predictions",
            "confirm raw 002 OOF score",
            "search class-wise bias on OOF",
            "save adjusted OOF proba as npy",
            "save adjusted test pred proba as npy",
            "save biased submission",
            "save bias vector and search results",
            "update oof_registry",
        ],
    },
    "data": {
        "train_path": str(TRAIN_PATH),
        "test_path": str(TEST_PATH),
        "sample_submission_path": str(SAMPLE_SUB_PATH),
        "target_col": TARGET_COL,
        "id_col": ID_COL,
        "raw_002_oof_path": str(oof_path),
        "raw_002_pred_path": str(pred_path),
        "use_original": False,
    },
    "bias_search": {
        "bias_space": {
            "GALAXY": "fixed_0",
            "QSO": "searched",
            "STAR": "searched",
        },
        "best_bias": {
            "GALAXY": float(best_bias[0]),
            "QSO": float(best_bias[1]),
            "STAR": float(best_bias[2]),
        },
        "raw_cv": float(raw_eval["balanced_accuracy"]),
        "biased_cv": float(biased_eval["balanced_accuracy"]),
        "delta_biased_minus_raw": float(biased_eval["balanced_accuracy"] - raw_eval["balanced_accuracy"]),
        "raw_eval": raw_eval,
        "biased_eval": biased_eval,
        "adjusted_probability_method": "softmax(log(raw_proba) + class_bias)",
    },
    "outputs": {
        "submission": "submission_lgb002_bias_search.csv",
        "bias_vector": "bias_vector.npy",
        "oof_proba": OOF_BIASED_NPY.name,
        "pred_proba": PRED_BIASED_NPY.name,
        "bias_search_results": "bias_search_results.csv",
        "raw_vs_biased_oof_eval": "raw_vs_biased_oof_eval.csv",
        "oof_csv": "oof_lgb002_bias_search.csv",
        "pred_csv": "pred_lgb002_bias_search.csv",
        "cv_result": "cv_result.json",
        "registry": "oof_registry.csv",
    },
    "judgement": {
        "status": "pending_public_lb",
        "reason": (
            "Bias search can improve OOF balanced accuracy, but may overfit class thresholds. "
            "Adoption depends on Public LB and blend behavior against raw 002/004/006."
        ),
        "next_action": [
            "Submit submission_lgb002_bias_search.csv",
            "Record Public LB",
            "Add oof_exp_20260603_007_lgb002_bias_search_proba.npy and pred_exp_20260603_007_lgb002_bias_search_proba.npy to blend notebook",
            "Compare against raw 002 and other baseline OOFs",
            "If blend does not improve, classify as risky_cv_only",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("registry saved:", registry_path)
print("memo saved:", memo_path)
display(registry.tail())

registry saved: /kaggle/working/oof_registry.csv
memo saved: /kaggle/working/exp_20260603_007_lgb002_bias_search/memo.yml


,exp_id,model_family,feature_family,cv_metric,cv_score,raw_cv_score,cv_delta_vs_raw,public_lb,use_original,use_fe,use_bias_search,oof_path,pred_path,submission_path,role,status,keep_hold_drop,notes
0,exp_20260603_007_lgb002_bias_search,LightGBM,strict_raw,balanced_accuracy_score,0.95861,0.956906,0.001704,NaN,False,False,True,/kaggle/working/exp_20260603_007_lgb002_bias_s...,/kaggle/working/exp_20260603_007_lgb002_bias_s...,/kaggle/working/exp_20260603_007_lgb002_bias_s...,postprocess_bias_candidate,completed,pending_public_lb,Class-wise bias search on 002 LGB raw. Blend-r...
